In [1]:
#!/usr/bin/env python
# coding: utf-8

from pathlib import Path
import os
import re
import json
import time
import hashlib
import random
import warnings
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.sparse import issparse

from llm_sc_curator import LLMscCurator
from llm_sc_curator.backends import GeminiBackend
from llm_sc_curator.noise_lists import NOISE_PATTERNS, NOISE_LISTS

from benchmarks.hierarchical_scoring import (
    score_hierarchical,
    _expected_major_state_generic,
    _parse_state_generic,
)
from benchmarks.cd8_config import CD8_HIER_CFG
from benchmarks.cd4_config import CD4_HIER_CFG
from benchmarks.caf_config import CAF_HIER_CFG
from benchmarks.mouse_b_config import MOUSE_B_CFG, score_mouse_b

from benchmarks.gt_mappings import (
    get_cd8_ground_truth,
    get_cd4_ground_truth,
    get_msc_ground_truth,
    get_bcell_ground_truth,
)

warnings.filterwarnings("ignore")

In [2]:
# =============================================================================
# 0. Reproducibility / paths
# =============================================================================
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

BASE = Path("/work")
INPUT_DIR = BASE / "paper" / "gb_resubmission" / "input"
OUTPUT_DIR = BASE / "paper" / "gb_resubmission" / "output" / "Fig2e_f_stepwise_comparable"
CACHE_DIR = OUTPUT_DIR / "llm_cache"
LONGFMT_DIR = OUTPUT_DIR / "stepwise_long_for_sanno"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
LONGFMT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# =============================================================================
# 1. .env loader
# =============================================================================
def parse_env_line(line: str):
    s = line.strip()
    if not s or s.startswith("#"):
        return None

    if s.lower().startswith("export "):
        s = s[7:].lstrip()

    if "=" not in s:
        return None

    key, value = s.split("=", 1)
    key = key.strip()

    def strip_inline_comment(val: str) -> str:
        in_single = False
        in_double = False
        for i, ch in enumerate(val):
            if ch == "'" and not in_double:
                in_single = not in_single
            elif ch == '"' and not in_single:
                in_double = not in_double
            elif ch == "#" and not in_single and not in_double:
                return val[:i].rstrip()
        return val

    value = strip_inline_comment(value.strip())

    if (value.startswith('"') and value.endswith('"')) or (value.startswith("'") and value.endswith("'")):
        quote = value[0]
        value = value[1:-1]
        if quote == '"':
            value = (
                value.replace(r"\n", "\n")
                     .replace(r"\r", "\r")
                     .replace(r"\t", "\t")
                     .replace(r"\\", "\\")
                     .replace(r"\"", "\"")
            )
    return key, value

def load_env_file_strict(path: str, override: bool = False):
    if not os.path.exists(path):
        raise FileNotFoundError(f".env not found: {path}")
    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            parsed = parse_env_line(raw)
            if not parsed:
                continue
            key, value = parsed
            if override:
                os.environ[key] = value
            else:
                os.environ.setdefault(key, value)

load_env_file_strict("/work/.env", override=False)

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    from getpass import getpass
    os.environ["GEMINI_API_KEY"] = getpass("Enter GEMINI_API_KEY (input hidden): ")
    GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]


In [ ]:
# =============================================================================
# 2. Backend / curator config
# =============================================================================
ACCURACY_MODE = True  # False: cheaper/faster, True: stronger but more expensive
MODEL_NAME = "models/gemini-2.5-pro" if ACCURACY_MODE else "models/gemini-2.0-flash"

if GEMINI_API_KEY is None or str(GEMINI_API_KEY).strip() == "":
    raise EnvironmentError("GEMINI_API_KEY is not set.")

# This flag controls whether full_core mirrors historical Fig.3 Curated behavior.
# True  -> full_core uses annotate(..., use_auto_context=True), closer to Fig.3 Curated
# False -> full_core uses annotate(..., use_auto_context=False), stricter feature-only stepwise
FULL_CORE_USE_AUTO_CONTEXT = True

# Recommended sleep to reduce quota bursts on flash models
API_SLEEP_SEC = 3.0

def make_curator(api_key: str, model_name: str):
    """
    Prefer the historical Fig.3-style constructor for comparability.
    Fallback to backend-based construction if needed by local package version.
    """
    try:
        curator = LLMscCurator(api_key=api_key, model_name=model_name)
        return curator
    except TypeError:
        backend = GeminiBackend(
            api_key=api_key,
            model_name=model_name,
            temperature=0.0,
        )
        curator = LLMscCurator(backend=backend)
        return curator

print(f"Using model: {MODEL_NAME}")
print(f"FULL_CORE_USE_AUTO_CONTEXT = {FULL_CORE_USE_AUTO_CONTEXT}")

In [ ]:
# =============================================================================
# 3. Dataset config
# =============================================================================
DATASETS = [
    {
        "name": "CD8",
        "task_key": "CD8",
        "path": INPUT_DIR / "cd8_benchmark_data.h5ad",
        "group_col": "meta.cluster",
        "ground_truth_col": "Ground_Truth",
        "ground_truth_fn": get_cd8_ground_truth,
        "n_top": 50,
        "exclude_gt": {"CD8_Other", "Other", "Unknown"},
    },

    # Uncomment if needed
    # {
    #     "name": "MSC",
    #     "task_key": "MSC",
    #     "path": INPUT_DIR / "msc_benchmark_data.h5ad",
    #     "group_col": "meta.cluster",
    #     "ground_truth_col": "Ground_Truth",
    #     "ground_truth_fn": get_msc_ground_truth,
    #     "n_top": 50,
    #     "exclude_gt": {"Other", "Unknown"},
    # },
]

VARIANT_ORDER = ["standard", "filter_only", "regex_mask", "full_core"]
VARIANT_DISPLAY = {
    "standard": "Standard",
    "filter_only": "filter_only",
    "regex_mask": "regex_mask",
    "full_core": "full_core",
}
VARIANT_COLOR = {
    "standard": "#5DA5DA",
    "filter_only": "#F0B43C",
    "regex_mask": "#60BD68",
    "full_core": "#B276B2",
}

In [ ]:
# =============================================================================
# 4. Task-specific Sanno configs
# =============================================================================
def _score_cd8(row, pred_col, cfg):
    return score_hierarchical(row, pred_col, cfg)

def _score_cd4(row, pred_col, cfg):
    return score_hierarchical(row, pred_col, cfg)

def _score_msc(row, pred_col, cfg):
    return score_caf_hierarchical(row, pred_col)

def _score_mouse_b(row, pred_col, cfg):
    return score_mouse_b(row, pred_col)

TASKS = {
    "CD8": {
        "cfg": CD8_HIER_CFG,
        "score_func": _score_cd8,
        "gt_mapper": get_cd8_ground_truth,
    },
    "CD4": {
        "cfg": CD4_HIER_CFG,
        "score_func": _score_cd4,
        "gt_mapper": get_cd4_ground_truth,
    },
    "MSC": {
        "cfg": CAF_HIER_CFG,
        "score_func": _score_msc,
        "gt_mapper": get_msc_ground_truth,
    },
    "MOUSE_B": {
        "cfg": MOUSE_B_CFG,
        "score_func": _score_mouse_b,
        "gt_mapper": get_bcell_ground_truth,
    },
}

In [ ]:
# =============================================================================
# 5. Noise helpers
# =============================================================================
COMPILED_REGEX_NOISE = {
    name: re.compile(pattern)
    for name, pattern in NOISE_PATTERNS.items()
}

CURATED_NOISE_GENE_SET = set()
for _, genes in NOISE_LISTS.items():
    CURATED_NOISE_GENE_SET.update(map(str, genes))

def is_regex_noise_gene(gene: str) -> bool:
    g = str(gene)
    return any(p.search(g) for p in COMPILED_REGEX_NOISE.values())

In [ ]:
# =============================================================================
# 5. Noise definitions / helpers
# =============================================================================
COMPILED_REGEX_NOISE = {
    name: re.compile(pattern)
    for name, pattern in NOISE_PATTERNS.items()
}

CURATED_NOISE_GENE_SET = set()
for _, genes in NOISE_LISTS.items():
    CURATED_NOISE_GENE_SET.update(map(str, genes))

def is_regex_noise_gene(gene: str) -> bool:
    g = str(gene)
    return any(p.search(g) for p in COMPILED_REGEX_NOISE.values())

In [ ]:
# =============================================================================
# 6. Basic DE / feature-selection helpers
# =============================================================================
def build_de_table(
    adata,
    group_col,
    cluster_name,
    min_target_mean=0.02,
    min_delta_mean=0.02,
    min_logfc=0.2,
    min_target_pct=0.02,
    min_delta_pct=0.02,
):
    tmp = "__tmp_binary__"
    adata.obs[tmp] = "Rest"
    adata.obs.loc[adata.obs[group_col].astype(str) == str(cluster_name), tmp] = "Target"

    sc.tl.rank_genes_groups(
        adata,
        groupby=tmp,
        groups=["Target"],
        reference="Rest",
        method="wilcoxon",
        use_raw=False,
    )
    de_df_raw = sc.get.rank_genes_groups_df(adata, group="Target").copy()

    target_mask = adata.obs[tmp] == "Target"
    rest_mask = adata.obs[tmp] == "Rest"

    X = adata.X
    if issparse(X):
        X_target = X[target_mask.values, :]
        X_rest = X[rest_mask.values, :]
        target_mean = np.asarray(X_target.mean(axis=0)).ravel()
        rest_mean = np.asarray(X_rest.mean(axis=0)).ravel()
        target_pct = np.asarray((X_target > 0).mean(axis=0)).ravel()
        rest_pct = np.asarray((X_rest > 0).mean(axis=0)).ravel()
    else:
        X_target = X[target_mask.values, :]
        X_rest = X[rest_mask.values, :]
        target_mean = X_target.mean(axis=0)
        rest_mean = X_rest.mean(axis=0)
        target_pct = (X_target > 0).mean(axis=0)
        rest_pct = (X_rest > 0).mean(axis=0)

    expr_stats = pd.DataFrame({
        "names": adata.var_names.astype(str),
        "target_mean": target_mean,
        "rest_mean": rest_mean,
        "target_pct": target_pct,
        "rest_pct": rest_pct,
    })

    de_df = de_df_raw.merge(expr_stats, on="names", how="left")
    de_df["delta_mean"] = de_df["target_mean"] - de_df["rest_mean"]
    de_df["delta_pct"] = de_df["target_pct"] - de_df["rest_pct"]

    eff_mask = (
        (de_df["target_mean"] >= min_target_mean) &
        (de_df["delta_mean"] >= min_delta_mean) &
        (de_df["target_pct"] >= min_target_pct) &
        (de_df["delta_pct"] >= min_delta_pct)
    )
    if "logfoldchanges" in de_df.columns:
        eff_mask &= (de_df["logfoldchanges"].fillna(0) >= min_logfc)

    de_df_filtered = de_df.loc[eff_mask].copy()
    adata.obs.drop(columns=[tmp], inplace=True, errors="ignore")
    return de_df_raw, de_df_filtered

def get_standard_deg_genes(de_df_raw, n_top=50):
    """
    Historical Fig.3-style Standard:
    raw ranked DEGs, not HVG-preferred.
    """
    return de_df_raw["names"].astype(str).head(n_top).tolist()

def get_filter_only_genes(de_df_raw, de_df_filtered, n_top=50):
    if de_df_filtered.empty:
        return de_df_raw["names"].astype(str).head(n_top).tolist()
    return de_df_filtered["names"].astype(str).head(n_top).tolist()

def get_regex_mask_genes(de_df_raw, de_df_filtered, n_top=50, oversample=300):
    """
    Effect-size filtered list + regex-only masking, no closed-set prompt tricks.
    """
    source_df = de_df_filtered if not de_df_filtered.empty else de_df_raw
    candidates = source_df["names"].astype(str).head(oversample).tolist()
    clean = [g for g in candidates if not is_regex_noise_gene(g)]
    return clean[:n_top]

def get_full_core_genes(curator, adata, group_col, cluster_name, n_top=50):
    """
    LLM-scCurator core feature distillation.
    HVGs may still be used internally by curate_features() as part of candidate-space construction,
    but NOT for Standard top-N generation.
    """
    genes = curator.curate_features(
        adata,
        group_col=group_col,
        target_group=str(cluster_name),
        n_top=n_top,
        use_statistics=True,
    )
    return [str(g) for g in genes[:n_top]]

In [ ]:
# =============================================================================
# 7. LLM annotate helpers (Fig.3-comparable open-ended mode)
# =============================================================================
def ensure_json_result(x):
    """
    Normalize annotate() output to a stable dict without forcing label normalization.
    """
    if isinstance(x, dict):
        return {
            "cell_type": x.get("cell_type", "Unknown"),
            "confidence": x.get("confidence", "Low"),
            "reasoning": x.get("reasoning", ""),
        }
    elif isinstance(x, str):
        return {
            "cell_type": x,
            "confidence": "Low",
            "reasoning": "",
        }
    else:
        return {
            "cell_type": "Error",
            "confidence": "Low",
            "reasoning": repr(x),
        }

def _cache_key(obj: dict) -> str:
    s = json.dumps(obj, sort_keys=True, ensure_ascii=False)
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def cached_annotate_call(
    curator,
    dataset_name: str,
    cluster_name: str,
    variant: str,
    genes: List[str],
    use_auto_context: bool,
    model_name: str,
    sleep_sec: float = API_SLEEP_SEC,
):
    payload = {
        "dataset": dataset_name,
        "cluster": cluster_name,
        "variant": variant,
        "genes": genes,
        "use_auto_context": bool(use_auto_context),
        "model_name": model_name,
        "annotation_mode": "open_ended_fig3_comparable",
    }

    key = _cache_key(payload)
    cache_path = CACHE_DIR / f"{key}.json"

    if cache_path.exists():
        with open(cache_path, "r", encoding="utf-8") as f:
            return json.load(f)

    try:
        raw_result = curator.annotate(genes, use_auto_context=use_auto_context)
        result = ensure_json_result(raw_result)
    except Exception as e:
        result = ensure_json_result(
            {"cell_type": "Error", "confidence": "Low", "reasoning": str(e)}
        )

    out = {
        "payload": payload,
        "result": result,
    }
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

    time.sleep(sleep_sec)
    return out

def annotate_variant_openended(
    curator,
    dataset_name: str,
    cluster_name: str,
    variant: str,
    genes: List[str],
    model_name: str,
):
    """
    Fig.3-comparable open-ended annotation.

    standard/filter_only/regex_mask -> use_auto_context=False
    full_core                       -> use_auto_context=FULL_CORE_USE_AUTO_CONTEXT
    """
    use_auto_context = (variant == "full_core" and FULL_CORE_USE_AUTO_CONTEXT)

    resp = cached_annotate_call(
        curator=curator,
        dataset_name=dataset_name,
        cluster_name=cluster_name,
        variant=variant,
        genes=genes,
        use_auto_context=use_auto_context,
        model_name=model_name,
    )
    res = resp["result"]

    return {
        "pred_text_for_scoring": res["cell_type"],
        "pred_confidence": res["confidence"],
        "pred_reasoning": res["reasoning"],
        "use_auto_context": use_auto_context,
        "annotate_raw_json": json.dumps(res, ensure_ascii=False),
    }


In [ ]:

# =============================================================================
# 8. Dataset preparation helpers
# =============================================================================
def ensure_ground_truth(adata, ds_cfg):
    gt_col = ds_cfg.get("ground_truth_col", "Ground_Truth")
    gt_fn = ds_cfg.get("ground_truth_fn", None)
    group_col = ds_cfg["group_col"]

    if gt_col not in adata.obs.columns:
        if gt_fn is None:
            raise ValueError(
                f"{ds_cfg['name']}: '{gt_col}' not found and no ground_truth_fn supplied."
            )
        adata.obs[gt_col] = adata.obs[group_col].apply(gt_fn)

    return gt_col

def ensure_hvgs(adata):
    """
    Keep HVG computation only because curate_features() may rely on it internally.
    Standard top-N does NOT use HVGs.
    """
    if "highly_variable" not in adata.var.columns:
        if "counts" in adata.layers:
            sc.pp.highly_variable_genes(
                adata,
                n_top_genes=2000,
                flavor="seurat_v3",
                layer="counts",
                subset=False,
            )
        else:
            sc.pp.highly_variable_genes(
                adata,
                n_top_genes=2000,
                flavor="seurat",
                subset=False,
            )

def get_cluster_list(adata, ds_cfg, gt_col):
    exclude_gt = set(ds_cfg.get("exclude_gt", []))
    group_col = ds_cfg["group_col"]

    clusters = sorted(adata.obs[group_col].astype(str).unique())
    keep = []
    for c in clusters:
        gt = adata.obs.loc[adata.obs[group_col].astype(str) == c, gt_col].iloc[0]
        if gt in exclude_gt:
            continue
        keep.append(c)
    return keep


In [ ]:
# =============================================================================
# 9. Sanno evaluation helpers
# =============================================================================
def bootstrap_mean_ci(values, n_boot=5000, seed=42, alpha=0.05):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return np.nan, np.nan
    if len(vals) == 1:
        return float(vals[0]), float(vals[0])

    rng = np.random.default_rng(seed)
    n = len(vals)
    boots = []
    for _ in range(n_boot):
        sample = rng.choice(vals, size=n, replace=True)
        boots.append(sample.mean())

    lo = float(np.quantile(boots, alpha / 2))
    hi = float(np.quantile(boots, 1 - alpha / 2))
    return lo, hi

def _build_state_series(df, cfg, pred_col):
    gt_states = []
    pred_states = []

    for _, row in df.iterrows():
        gt_major, gt_state = _expected_major_state_generic(row["Ground_Truth"], cfg)
        if gt_state == cfg.default_state:
            continue
        pred_state = _parse_state_generic(str(row[pred_col]), cfg)
        gt_states.append(gt_state)
        pred_states.append(pred_state)

    if not gt_states:
        return None, None

    gt_series = pd.Series(gt_states, name="GT_state")
    pred_series = pd.Series(pred_states, name="Pred_state")
    return gt_series, pred_series

def _confusion_and_per_state_metrics(gt, pred):
    cm = pd.crosstab(gt, pred)
    labels = sorted(set(gt.unique()) | set(pred.unique()))
    rows = []

    for lab in labels:
        tp = cm.loc[lab, lab] if (lab in cm.index and lab in cm.columns) else 0
        support = int(cm.loc[lab].sum()) if lab in cm.index else 0
        fp = int(cm[lab].sum() - tp) if lab in cm.columns else 0
        fn = support - tp

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2.0 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

        rows.append({
            "State": lab,
            "Support": support,
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
        })

    metrics_df = pd.DataFrame(rows).sort_values("State")
    return cm, metrics_df

def score_dataset_stepwise(df_ds, task_key, pred_col="Pred_Text"):
    task_spec = TASKS[task_key]
    cfg = task_spec["cfg"]
    score_func = task_spec["score_func"]

    df_scored = df_ds.copy()
    df_scored["Sanno"] = df_scored.apply(lambda row: score_func(row, pred_col, cfg), axis=1)

    gt_major = []
    gt_state = []
    used_in_cm = []
    for gt_label in df_scored["Ground_Truth"]:
        major, state = _expected_major_state_generic(gt_label, cfg)
        gt_major.append(major)
        gt_state.append(state)
        used_in_cm.append(state != cfg.default_state)

    df_scored["GT_Major"] = gt_major
    df_scored["GT_State"] = gt_state
    df_scored["UsedInConfusion"] = used_in_cm

    summary_rows = []
    for variant in VARIANT_ORDER:
        sub = df_scored[df_scored["Variant"] == variant].copy()
        if sub.empty:
            continue

        scores = sub["Sanno"].astype(float)
        mean_score = float(scores.mean())
        frac_eq1 = float((scores == 1.0).mean())
        frac_ge05 = float((scores >= 0.5).mean())
        ci_lo, ci_hi = bootstrap_mean_ci(scores.values, n_boot=5000, seed=RANDOM_SEED)

        summary_rows.append({
            "Dataset": sub["Dataset"].iloc[0],
            "Variant": variant,
            "N": int(len(scores)),
            "MeanScore": mean_score,
            "FracScoreEq1": frac_eq1,
            "FracScoreGe0_5": frac_ge05,
            "MeanScore_CI_Lo": ci_lo,
            "MeanScore_CI_Hi": ci_hi,
        })

        gt_series, pred_series = _build_state_series(sub, cfg, pred_col)
        if gt_series is not None:
            cm, metrics_df = _confusion_and_per_state_metrics(gt_series, pred_series)

            ds_name = sub["Dataset"].iloc[0]
            cm_path = OUTPUT_DIR / f"{ds_name}_confusion_{variant}.csv"
            metrics_path = OUTPUT_DIR / f"{ds_name}_per_state_metrics_{variant}.csv"

            cm.to_csv(cm_path)
            metrics_df.to_csv(metrics_path, index=False)

            print(f"[INFO] {ds_name}/{variant}: confusion → {cm_path}")
            print(f"[INFO] {ds_name}/{variant}: per-state metrics → {metrics_path}")

    return df_scored, pd.DataFrame(summary_rows)


In [ ]:
# =============================================================================
# 10. Run predictions
# =============================================================================
all_rows = []

for ds_cfg in DATASETS:
    ds_name = ds_cfg["name"]
    task_key = ds_cfg["task_key"]

    print(f"\n{'='*80}\nRunning dataset: {ds_name}\n{'='*80}")

    adata = sc.read_h5ad(ds_cfg["path"])
    gt_col = ensure_ground_truth(adata, ds_cfg)

    # Keep HVGs only for curate_features(), not for Standard top-N
    ensure_hvgs(adata)

    group_col = ds_cfg["group_col"]
    n_top = int(ds_cfg.get("n_top", 50))

    cluster_list = get_cluster_list(adata, ds_cfg, gt_col)

    # Example pilot subset:
    #cluster_list = [c for c in cluster_list if c in {
    #     "CD8.c01.Tn.MAL",
    #     "CD8.c02.Tm.IL7R",
    #     "CD8.c06.Tem.GZMK",
    #     "CD8.c12.Tex.CXCL13",
    #     "CD8.c16.MAIT.SLC4A10",
    #}]

    curator = make_curator(GEMINI_API_KEY, MODEL_NAME)
    curator.set_global_context(adata)

    for i, cluster_name in enumerate(cluster_list, start=1):
        gt_label = (
            adata.obs.loc[adata.obs[group_col].astype(str) == str(cluster_name), gt_col]
            .astype(str)
            .iloc[0]
        )
        print(f"[{ds_name} {i}/{len(cluster_list)}] {cluster_name} -> {gt_label}")

        de_df_raw, de_df_filtered = build_de_table(
            adata=adata,
            group_col=group_col,
            cluster_name=cluster_name,
        )

        genes_standard = get_standard_deg_genes(de_df_raw, n_top=n_top)
        genes_filter = get_filter_only_genes(de_df_raw, de_df_filtered, n_top=n_top)
        genes_regex = get_regex_mask_genes(de_df_raw, de_df_filtered, n_top=n_top, oversample=300)
        genes_core = get_full_core_genes(curator, adata, group_col, cluster_name, n_top=n_top)

        variant_to_genes = {
            "standard": genes_standard,
            "filter_only": genes_filter,
            "regex_mask": genes_regex,
            "full_core": genes_core,
        }

        for variant in VARIANT_ORDER:
            genes = variant_to_genes[variant]

            pred = annotate_variant_openended(
                curator=curator,
                dataset_name=ds_name,
                cluster_name=str(cluster_name),
                variant=variant,
                genes=genes,
                model_name=MODEL_NAME,
            )

            all_rows.append({
                "Dataset": ds_name,
                "TaskKey": task_key,
                "Cluster_ID": str(cluster_name),
                "Ground_Truth": gt_label,
                "Variant": variant,
                "Pred_Text": pred["pred_text_for_scoring"],
                "Pred_Confidence": pred["pred_confidence"],
                "Pred_Reasoning": pred["pred_reasoning"],
                "Use_Auto_Context": pred["use_auto_context"],
                "N_Input_Genes": len(genes),
                "Genes": "|".join(genes),
                "Annotate_Raw_JSON": pred["annotate_raw_json"],
            })


In [ ]:
# =============================================================================
# 11. Save raw long-format predictions
# =============================================================================
df_all = pd.DataFrame(all_rows)
raw_long_path = OUTPUT_DIR / "stepwise_end_to_end_cluster_predictions_long.csv"
df_all.to_csv(raw_long_path, index=False)
print(f"\n[INFO] Raw long-format predictions → {raw_long_path}")

file_map = {
    "CD8": "cd8_stepwise_long.csv",
    "CD4": "cd4_stepwise_long.csv",
    "MSC": "msc_stepwise_long.csv",
    "MOUSE_B": "mouse_b_stepwise_long.csv",
}
for ds_name, fn in file_map.items():
    sub = df_all[df_all["Dataset"] == ds_name].copy()
    if not sub.empty:
        out_path = LONGFMT_DIR / fn
        sub.to_csv(out_path, index=False)
        print(f"[INFO] Saved {ds_name} long-format → {out_path}")

In [ ]:
# =============================================================================
# 12. S_anno evaluation
# =============================================================================
scored_dfs = []
summary_dfs = []

for ds_cfg in DATASETS:
    ds_name = ds_cfg["name"]
    task_key = ds_cfg["task_key"]

    sub = df_all[df_all["Dataset"] == ds_name].copy()
    if sub.empty:
        continue

    df_scored, df_summary = score_dataset_stepwise(
        df_ds=sub,
        task_key=task_key,
        pred_col="Pred_Text",
    )

    scored_path = OUTPUT_DIR / f"{ds_name.lower()}_stepwise_scored.csv"
    summary_path = OUTPUT_DIR / f"{ds_name.lower()}_stepwise_summary.csv"

    df_scored.to_csv(scored_path, index=False)
    df_summary.to_csv(summary_path, index=False)

    print(f"[INFO] {ds_name}: scored CSV → {scored_path}")
    print(f"[INFO] {ds_name}: summary CSV → {summary_path}")
    print(df_summary)

    scored_dfs.append(df_scored)
    summary_dfs.append(df_summary)

df_scored_all = pd.concat(scored_dfs, axis=0, ignore_index=True) if len(scored_dfs) > 0 else pd.DataFrame()
df_summary_all = pd.concat(summary_dfs, axis=0, ignore_index=True) if len(summary_dfs) > 0 else pd.DataFrame()

global_summary_path = OUTPUT_DIR / "Fig2e_data.csv"
df_summary_all.to_csv(global_summary_path, index=False)
print(f"[INFO] Global stepwise summary → {global_summary_path}")


In [ ]:
# =============================================================================
# 13. Plot Mean Sanno with bootstrap CI
# =============================================================================
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "DejaVu Sans"]

datasets_in_plot = list(df_summary_all["Dataset"].unique())
n_panels = len(datasets_in_plot)

if n_panels > 0:
    fig, axes = plt.subplots(
        1, n_panels,
        figsize=(5.4 * n_panels, 4.4),
        dpi=300,
        squeeze=False
    )
    axes = axes.ravel()

    for ax, ds_name in zip(axes, datasets_in_plot):
        sub = df_summary_all[df_summary_all["Dataset"] == ds_name].copy()
        sub["Variant"] = pd.Categorical(sub["Variant"], categories=VARIANT_ORDER, ordered=True)
        sub = sub.sort_values("Variant")

        x = np.arange(len(sub))
        vals = sub["MeanScore"].values * 100
        err_lo = (sub["MeanScore"] - sub["MeanScore_CI_Lo"]).values * 100
        err_hi = (sub["MeanScore_CI_Hi"] - sub["MeanScore"]).values * 100

        ax.bar(
            x,
            vals,
            color=[VARIANT_COLOR[v] for v in sub["Variant"]],
            edgecolor="black",
            linewidth=1.0,
        )
        ax.errorbar(
            x,
            vals,
            yerr=[err_lo, err_hi],
            fmt="none",
            ecolor="black",
            elinewidth=1.0,
            capsize=3,
        )

        for i, v in enumerate(vals):
            ax.text(i, v + 1.2, f"{v:.1f}", ha="center", va="bottom", fontsize=9)

        ax.set_xticks(x)
        ax.set_xticklabels([VARIANT_DISPLAY[v] for v in sub["Variant"]], rotation=35, ha="right")
        ax.set_ylabel("Mean S_anno (%)", fontweight="bold")
        ax.set_xlabel(ds_name, fontweight="bold")
        ax.set_ylim(0, max(100, np.nanmax(vals) + 10))
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "Fig2e.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / "Fig2e.pdf", bbox_inches="tight")
    plt.show()

In [ ]:
# =============================================================================
# 14. Console summary
# =============================================================================
print("\n==============================")
print("Stepwise end-to-end Sanno summary")
print("==============================")
print(df_summary_all)

In [ ]:
# =============================================================================
# 15. Refined stepwise overview for editor/reviewer response
#     - mean Sanno bar plot
#     - cluster x variant Sanno heatmap
#     - keep source-style CSV outputs unchanged
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------------------------------------------------------
# Fallback load if needed
# -------------------------------------------------------------------------
if "df_scored_all" not in globals() or df_scored_all is None or len(df_scored_all) == 0:
    scored_files = sorted(Path(OUTPUT_DIR).glob("*_stepwise_scored.csv"))
    if len(scored_files) == 0:
        raise FileNotFoundError("No *_stepwise_scored.csv files found in OUTPUT_DIR.")
    df_scored_all = pd.concat([pd.read_csv(p) for p in scored_files], ignore_index=True)

if "df_summary_all" not in globals() or df_summary_all is None or len(df_summary_all) == 0:
    summary_files = sorted(Path(OUTPUT_DIR).glob("*_stepwise_summary.csv"))
    if len(summary_files) == 0:
        raise FileNotFoundError("No *_stepwise_summary.csv files found in OUTPUT_DIR.")
    df_summary_all = pd.concat([pd.read_csv(p) for p in summary_files], ignore_index=True)

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "DejaVu Sans"]


# -------------------------------------------------------------------------
# Helper: recompute summary from scored table
# -------------------------------------------------------------------------
def recompute_summary_from_scored(df_scored, variant_order):
    rows = []
    for variant in variant_order:
        sub = df_scored[df_scored["Variant"] == variant].copy()
        if sub.empty:
            continue

        scores = sub["Sanno"].astype(float).values
        mean_score = float(np.mean(scores))
        frac_eq1 = float(np.mean(scores == 1.0))
        frac_ge05 = float(np.mean(scores >= 0.5))

        rng = np.random.default_rng(42)
        boots = []
        n = len(scores)
        for _ in range(5000):
            samp = rng.choice(scores, size=n, replace=True)
            boots.append(np.mean(samp))
        ci_lo = float(np.quantile(boots, 0.025))
        ci_hi = float(np.quantile(boots, 0.975))

        rows.append({
            "Dataset": sub["Dataset"].iloc[0],
            "Variant": variant,
            "N": int(n),
            "MeanScore": mean_score,
            "FracScoreEq1": frac_eq1,
            "FracScoreGe0_5": frac_ge05,
            "MeanScore_CI_Lo": ci_lo,
            "MeanScore_CI_Hi": ci_hi,
        })
    return pd.DataFrame(rows)


# -------------------------------------------------------------------------
# Helper: build cluster matrix and delta tables
# -------------------------------------------------------------------------
def build_stepwise_tables(df_scored_ds, variant_order):
    cluster_mat = (
        df_scored_ds.pivot_table(
            index="Cluster_ID",
            columns="Variant",
            values="Sanno",
            aggfunc="first"
        )
        .reindex(columns=variant_order)
    )

    gt_map = (
        df_scored_ds[["Cluster_ID", "Ground_Truth"]]
        .drop_duplicates()
        .set_index("Cluster_ID")["Ground_Truth"]
        .to_dict()
    )

    # order by full_core benefit over standard if available
    if "standard" in cluster_mat.columns and "full_core" in cluster_mat.columns:
        cluster_order = (
            (cluster_mat["full_core"] - cluster_mat["standard"])
            .sort_values(ascending=False)
            .index.tolist()
        )
        cluster_mat = cluster_mat.loc[cluster_order]

    df_cluster_matrix = cluster_mat.reset_index().copy()
    df_cluster_matrix.insert(
        1,
        "Ground_Truth",
        df_cluster_matrix["Cluster_ID"].map(gt_map)
    )

    delta_rows = []
    if "standard" in cluster_mat.columns:
        for variant in variant_order:
            if variant == "standard":
                continue
            for cluster_id in cluster_mat.index:
                base = cluster_mat.loc[cluster_id, "standard"]
                alt = cluster_mat.loc[cluster_id, variant]
                if pd.isna(base) or pd.isna(alt):
                    continue
                delta_rows.append({
                    "Dataset": df_scored_ds["Dataset"].iloc[0],
                    "Cluster_ID": cluster_id,
                    "Ground_Truth": gt_map.get(cluster_id, "NA"),
                    "Variant": variant,
                    "Standard_S_anno": float(base),
                    "Variant_Sanno": float(alt),
                    "Delta_vs_Standard": float(alt - base),
                })

    df_delta = pd.DataFrame(delta_rows)

    delta_summary_rows = []
    if not df_delta.empty:
        for variant in [v for v in variant_order if v != "standard"]:
            sub = df_delta[df_delta["Variant"] == variant].copy()
            if sub.empty:
                continue

            improved = int((sub["Delta_vs_Standard"] > 0).sum())
            unchanged = int(np.isclose(sub["Delta_vs_Standard"], 0.0, atol=1e-12).sum())
            worsened = int((sub["Delta_vs_Standard"] < 0).sum())

            delta_summary_rows.append({
                "Dataset": sub["Dataset"].iloc[0],
                "Variant": variant,
                "N_Clusters": int(len(sub)),
                "Mean_Delta_vs_Standard": float(sub["Delta_vs_Standard"].mean()),
                "Median_Delta_vs_Standard": float(sub["Delta_vs_Standard"].median()),
                "N_Improved": improved,
                "N_Unchanged": unchanged,
                "N_Worsened": worsened,
            })

    df_delta_summary = pd.DataFrame(delta_summary_rows)

    return cluster_mat, gt_map, df_cluster_matrix, df_delta, df_delta_summary


# -------------------------------------------------------------------------
# Main overview plot: two panels only
# -------------------------------------------------------------------------
def plot_stepwise_overview_for_dataset(df_scored_ds, dataset_name, outdir):
    variant_order = [v for v in VARIANT_ORDER if v in df_scored_ds["Variant"].unique()]
    if len(variant_order) == 0:
        return

    df_summary_ds = recompute_summary_from_scored(df_scored_ds, variant_order)
    cluster_mat, gt_map, df_cluster_matrix, df_delta, df_delta_summary = build_stepwise_tables(
        df_scored_ds=df_scored_ds,
        variant_order=variant_order,
    )

    # Keep source-style CSV outputs unchanged
    df_cluster_matrix.to_csv(outdir / f"Fig2f_{dataset_name}_stepwise_cluster_matrix.csv", index=False)
    df_delta.to_csv(outdir / f"{dataset_name}_stepwise_delta_vs_standard.csv", index=False)
    df_delta_summary.to_csv(outdir / f"{dataset_name}_stepwise_delta_summary.csv", index=False)

    fig, (ax1, ax2) = plt.subplots(
        1, 2,
        figsize=(12.8, 4.8),
        dpi=300,
        gridspec_kw={"width_ratios": [1.0, 1.25]}
    )

    # -----------------------------------------------------------------
    # Fig2e: Mean Sanno
    # -----------------------------------------------------------------
    sub = df_summary_ds.copy()
    sub["Variant"] = pd.Categorical(sub["Variant"], categories=variant_order, ordered=True)
    sub = sub.sort_values("Variant")

    x = np.arange(len(sub))
    vals = sub["MeanScore"].values * 100
    err_lo = (sub["MeanScore"] - sub["MeanScore_CI_Lo"]).values * 100
    err_hi = (sub["MeanScore_CI_Hi"] - sub["MeanScore"]).values * 100

    ax1.bar(
        x,
        vals,
        color=[VARIANT_COLOR[v] for v in sub["Variant"]],
        edgecolor="black",
        linewidth=1.0,
    )
    ax1.errorbar(
        x,
        vals,
        yerr=[err_lo, err_hi],
        fmt="none",
        ecolor="black",
        elinewidth=1.0,
        capsize=3,
    )

    for i, v in enumerate(vals):
        ax1.text(i, v + 1.0, f"{v:.1f}", ha="center", va="bottom", fontsize=12, fontweight="bold")

    ax1.set_xticks(x)
    ax1.set_xticklabels([VARIANT_DISPLAY[v] for v in sub["Variant"]], rotation=30, ha="right")
    ax1.set_ylabel("Mean S_anno (%)", fontweight="bold")
    ax1.set_xlabel(f"{dataset_name}: mean ontology-aware accuracy", fontweight="bold")
    ax1.set_ylim(0, max(100, np.nanmax(vals) + 12))
    ax1.spines["top"].set_visible(False)
    ax1.spines["right"].set_visible(False)

    # -----------------------------------------------------------------
    # Fig2f: Heatmap cluster x variant
    # -----------------------------------------------------------------
    heat = cluster_mat[variant_order].copy()
    im = ax2.imshow(heat.values, aspect="auto", vmin=0, vmax=1, cmap="YlGnBu")

    ax2.set_xticks(range(len(variant_order)))
    ax2.set_xticklabels([VARIANT_DISPLAY[v] for v in variant_order], rotation=30, ha="right")
    ax2.set_yticks(range(len(heat.index)))
    ax2.set_yticklabels(
        [f"{cid} | {gt_map.get(cid, 'NA')}" for cid in heat.index],
        fontsize=7,
        fontweight="bold"
    )
    ax2.set_xlabel(f"{dataset_name}: cluster-by-variant S_anno", fontweight="bold")

    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            val = heat.iloc[i, j]
            if pd.isna(val):
                continue
            txt_color = "white" if val >= 0.5 else "black"
            ax2.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7, color=txt_color)

    cbar = fig.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
    cbar.set_label("S_anno")

    fig.tight_layout()
    fig.savefig(outdir / f"Fig2e_f_{dataset_name}_stepwise_overview.png", dpi=300, bbox_inches="tight")
    fig.savefig(outdir / f"Fig2e_f_{dataset_name}_stepwise_overview.pdf", bbox_inches="tight")
    plt.close(fig)


# -------------------------------------------------------------------------
# Run all refined overview visualizations
# -------------------------------------------------------------------------
for ds_name in sorted(df_scored_all["Dataset"].unique()):
    df_scored_ds = df_scored_all[df_scored_all["Dataset"] == ds_name].copy()
    if df_scored_ds.empty:
        continue

    plot_stepwise_overview_for_dataset(
        df_scored_ds=df_scored_ds,
        dataset_name=ds_name,
        outdir=OUTPUT_DIR,
    )

print("[INFO] Refined two-panel stepwise overview visualizations completed.")